# Import libraries and set variables

In [6]:
import csv
import json
import os
import pandas as pd
from pathlib import Path
import platform
import re
import requests
import sys
from time import sleep
from tqdm import tqdm
tqdmBarFormat = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions
from dataverse_functions import dataverse_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

rorRegex = r'https?://ror\.org/0[a-hj-km-np-tv-z|0-9]{6}[0-9]{2}$'
processedDataDir = '/Users/juliangautier/ux-research-data-analysis/Findability_of_author_affiliations_with_rors/data/processed'

# Measure bug's affect on findability of datasets in Dataverse installations

In [7]:
# Get names and ROR URLs of author affiliations in datasets published in each Dataverse installation
rorUrlPrenset_affNameNotPresentDf = pd.read_csv(
    os.path.join(processedDataDir, 'rorUrlPrenset_affNameNotPresent.csv'),
    encoding='utf-8-sig')

rorUrlPrenset_affNameNotPresentDf.head()

,dataverse_installation_name,dataverse_installation_url,ror_display_name_2026_11_01,author_affiliation_ror,expanded_value_key_present
0,DANS Data Station Archaeology,https://archaeology.datastations.nl,Cranfield University,['https://ror.org/05cncd958'],False
1,DANS Data Station Archaeology,https://archaeology.datastations.nl,Digital Heritage Netherlands,['https://ror.org/055622428'],False
2,DANS Data Station Archaeology,https://archaeology.datastations.nl,Gemeente Den Haag,['https://ror.org/03w387231'],True
3,DANS Data Station Archaeology,https://archaeology.datastations.nl,Gemeente Eindhoven,['https://ror.org/04dyx6987'],False
4,DANS Data Station Archaeology,https://archaeology.datastations.nl,National Museum of Antiquities,['https://ror.org/01079a392'],False


In [ ]:
# For each affiliation name, do a name-based search and a ROR-based search for each ROR

for _, installationDf in rorUrlPrenset_affNameNotPresentDf.groupby(['dataverse_installation_url']):
    installationName = installationDf['dataverse_installation_name'].iloc[0]
    installation_url = installationDf['dataverse_installation_url'].iloc[0]
    safeInstallationName = common_functions.safe_filename(installationName)
    
    if installationName == 'Harvard Dataverse':
        rootCollectionInfoDict = dataverse_functions.get_root_collection_info(installation_url)
        rootAlias = rootCollectionInfoDict['rootAlias']
        for dfIndex, dfRow in installationDf.iterrows():
            authorAffiliationName = dfRow['ror_display_name_2026_11_01']
            authorAffiliationRorUrlList = common_functions.string_to_list(dfRow['author_affiliation_ror'])

            if authorAffiliationName == 'Philipps University of Marburg':

                # Create directory for affiliaton search results
                installationDirPath = os.path.join(desktopPath, 'affiliation_search_results', safeInstallationName)
                os.makedirs(installationDirPath)
              
                # Use Search API to get list of datasets returned by search of author affiliation name
                print('Searching for affiliation by name...')
                nameBasedSearchUrl = f'{installation_url}/dataverse/{rootAlias}/?q=authorAffiliation"{authorAffiliationName}"'
                print(nameBasedSearchUrl)

                sleep(.5)
                nameBasedSearchResultsDf = dataverse_functions.get_datasets_from_collection_or_search_url(
                    nameBasedSearchUrl, headers={}, apiKey='', 
                    metadataFieldsParam=['citation:author'],
                    ignoreDeaccessionedDatasets=True,
                    includeSubDataverses=True, includeLinkedDatasets=True, 
                    includeHarvestedDatasets=False,
                    printProgress=True)

                if nameBasedSearchResultsDf is not None:

                    # Reorder columns
                    nameBasedSearchResultsDf = nameBasedSearchResultsDf[[
                        'dataset_pid', 'dataverse_collection_alias', 
                        'latest_version', 'dataset_last_updated', 'citation:author'
                    ]]

                    # Export search results
                    filePath = os.path.join(installationDirPath,  f'name_based_search_results.csv')
                    nameBasedSearchResultsDf.to_csv(filePath, index=False,  encoding='utf-8-sig')
                
                sleep(.5)

                # Use Search API to get list of datasets returned by search of ROR URLs
                rorBasedSearchResultsDfList = []
                print('Searching for affiliation by ROR URL...')
                for rorUrl in authorAffiliationRorUrlList:
                    rorBasedSearchUrl = f'{installation_url}/dataverse/{rootAlias}/?q=authorAffiliation"{rorUrl}"'
                    print(rorBasedSearchUrl)

                    rorBasedSearchResultsDf = dataverse_functions.get_datasets_from_collection_or_search_url(
                        rorBasedSearchUrl, headers={}, apiKey='', 
                        metadataFieldsParam=['citation:author'],
                        ignoreDeaccessionedDatasets=True,
                        includeSubDataverses=True, includeLinkedDatasets=True, 
                        includeHarvestedDatasets=False,
                        printProgress=True)

                    if rorBasedSearchResultsDf is not None:
                        rorBasedSearchResultsDfList.append(rorBasedSearchResultsDf)

                if len(rorBasedSearchResultsDfList) > 0:
                    # Concat dataframes in rorBasedSearchResultsDfList
                    rorBasedSearchDf = pd.concat(rorBasedSearchResultsDfList)

                    # Reorder columns
                    rorBasedSearchResultsDf = rorBasedSearchResultsDf[[
                        'dataset_pid', 'dataverse_collection_alias', 
                        'latest_version', 'dataset_last_updated', 'citation:author'
                    ]]

                    # Export search results
                    filePath = os.path.join(installationDirPath,  f'ror_based_search_results.csv')
                    rorBasedSearchResultsDf.to_csv(filePath, index=False,  encoding='utf-8-sig')


            # print(authorAffiliationName)

Searching for affiliation by name...
https://dataverse.harvard.edu/dataverse/harvard/?q=authorAffiliation"Philipps University of Marburg"


100%|██████████| 1/1 [Elapsed:00:01<Remaining:00:00, 1.63s/it, Page 1]


Distinct datasets found: 10
Searching for affiliation by ROR URL...
https://dataverse.harvard.edu/dataverse/harvard/?q=authorAffiliation"https://ror.org/01rdrb571"


100%|██████████| 2/2 [Elapsed:00:02<Remaining:00:00, 1.50s/it, Page 11]

Distinct datasets found: 12


In [21]:
rorUrl = 'https://ror.org/04e27p903'
rorId = rorUrl.split('/')[-1]
print(rorId)

04e27p903
